# 04 — RMT Eigenvalue Analysis

For each saved frame the 50×50 ZEB field is symmetrised and its eigenvalue distribution estimated via Gaussian KDE.

- Early frames: broad near-semicircular distribution (disordered)
- Late frames: outlier λ_max separates as M-state domain expands

> Run notebook 03 first to generate `results/data/frames_I50k.npy`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from ipywidgets import Play, IntSlider, jslink, interact
from IPython.display import display
from google.colab import output
output.enable_custom_widget_manager()

from src.analysis.rmt_analysis import compute_eigenvalue_pdfs

print("Setup complete.")


## Load frames

In [ ]:
frames     = np.load('../results/data/frames_I50k.npy')
SAVE_EVERY = 20
DT         = 0.05
print(f"Loaded: {frames.shape}  →  {frames.shape[0]} frames, {frames.shape[1]}×{frames.shape[2]} grid")


## Precompute eigenvalues & KDE (~1–2 min)

In [ ]:
print("Precomputing...")
eigvals, x_grid, all_kde = compute_eigenvalue_pdfs(
    frames, species_idx=2, bw_method=0.15, n_grid=500
)

n_frames     = frames.shape[0]
time_axis    = np.arange(n_frames) * SAVE_EVERY * DT
y_global_max = all_kde.max() * 1.05

print(f"Done. λ range: [{eigvals.min():.2f}, {eigvals.max():.2f}]")


## Animated eigenvalue PDF widget

In [ ]:
def plot_eigdist(idx):
    ev    = eigvals[idx]
    kde_y = all_kde[idx]
    t     = time_axis[idx]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5),
                              gridspec_kw={'width_ratios': [2, 1]})
    fig.patch.set_facecolor('#0a0a14')

    for ax in axes:
        ax.set_facecolor('#0a0a14')
        ax.tick_params(colors='white')
        for sp in ax.spines.values():
            sp.set_edgecolor('#333')
        ax.xaxis.label.set_color('white')
        ax.yaxis.label.set_color('white')
        ax.title.set_color('white')

    # Left: KDE PDF
    ax = axes[0]
    ax.fill_between(x_grid, kde_y, where=x_grid <  0, color='#ef5350', alpha=0.4, label='Negative λ')
    ax.fill_between(x_grid, kde_y, where=x_grid >= 0, color='#4fc3f7', alpha=0.4, label='Positive λ')
    ax.plot(x_grid, kde_y, color='white', lw=1.5)
    ax.axvline(0,      color='white',   ls='--', lw=1.0, alpha=0.5)
    ax.axvline(ev[-1], color='#ffcc80', ls='--', lw=1.2, label=f'λ_max = {ev[-1]:.1f}')
    ax.axvline(ev[0],  color='#ef9a9a', ls='--', lw=1.2, label=f'λ_min = {ev[0]:.1f}')
    ax.set_xlim(x_grid[0], x_grid[-1])
    ax.set_ylim(0, y_global_max)
    ax.set_xlabel('Eigenvalue λ'); ax.set_ylabel('Probability density')
    ax.set_title(f'ZEB matrix eigenvalue PDF  |  t = {t:.1f}  |  λ_max = {ev[-1]:.1f},  λ_min = {ev[0]:.1f}', fontsize=10)
    ax.legend(fontsize=8, framealpha=0.15, labelcolor='white', loc='upper left')

    # Right: λ_max / λ_min tracker
    ax2 = axes[1]
    ax2.plot(time_axis,        eigvals[:, -1], color='#2a2a3a', lw=1.5, zorder=1)
    ax2.plot(time_axis[:idx+1], eigvals[:idx+1, -1], color='#4fc3f7', lw=1.8, zorder=2, label='λ_max')
    ax2.plot(time_axis,        eigvals[:,  0], color='#1a1a2a', lw=1.5, zorder=1)
    ax2.plot(time_axis[:idx+1], eigvals[:idx+1,  0], color='#ef5350', lw=1.8, zorder=2, label='λ_min')
    ax2.scatter([t], [ev[-1]], color='#ffcc80', s=55, zorder=3)
    ax2.scatter([t], [ev[0]],  color='#ef9a9a', s=55, zorder=3)
    ax2.axhline(0, color='white', ls=':', lw=0.8, alpha=0.4)
    ax2.set_xlabel('Time'); ax2.set_ylabel('Eigenvalue')
    ax2.set_title('λ_max and λ_min vs time')
    ax2.legend(fontsize=7.5, framealpha=0.15, labelcolor='white')

    plt.tight_layout(); plt.show()


play   = Play(value=0, min=0, max=n_frames-1, step=1, interval=150, description="Play")
slider = IntSlider(value=0, min=0, max=n_frames-1, step=1,
                   description='Frame',
                   style={'description_width': 'initial'},
                   layout={'width': '500px'})

jslink((play, 'value'), (slider, 'value'))
interact(plot_eigdist, idx=slider)
display(play)
